# Fase 1 · Paso 4: repetición controlada con `tibiofemoral_crop_v0.4_pilot`

Esta libreta verifica el cierre rechazado de `v0.3`, ejecuta `v0.4` únicamente sobre las mismas veinte rodillas del piloto y muestra las nuevas vistas para una revisión visual ciega. No procesa la cohorte completa, no crea particiones, no entrena y no abre la prueba reservada.

## 1. Montar Drive y verificar las evidencias previas

In [ ]:
import json
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT antes de continuar.')
os.environ['KNEE_DATA_ROOT'] = private_root
private_root_path = Path(private_root)
bilateral_dir = private_root_path / 'outputs' / 'auditorias' / 'fase_1' / 'paso_3_separacion_bilateral'
step4_dir = private_root_path / 'outputs' / 'auditorias' / 'fase_1' / 'paso_4_localizacion_tibiofemoral'
v03_dir = step4_dir / 'v0_3_piloto'
v03_summary_path = v03_dir / 'resumen_revision_visual_publico.json'
v03_record_path = v03_dir / 'registro_cierre_revision.json'
if not (bilateral_dir / 'cierre_paso_3_publico.json').is_file():
    raise FileNotFoundError('No se encontró el cierre del paso 3.')
bilateral_closure = json.loads((bilateral_dir / 'cierre_paso_3_publico.json').read_text(encoding='utf-8'))
if bilateral_closure.get('status') != 'closed' or not bilateral_closure.get('parameters_frozen'):
    raise RuntimeError('El paso 3 no está cerrado y congelado.')
if not v03_summary_path.is_file() or not v03_record_path.is_file():
    raise FileNotFoundError('No se encontró el cierre reproducible de v0.3.')
print('Drive montado y evidencias previas localizadas.')

## 2. Obtener una revisión identificable del repositorio

In [ ]:
import subprocess

REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_fase1_paso4_v04')
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('La ruta de trabajo existe y no es un clon Git; usa un entorno nuevo.')
    dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('El clon temporal contiene cambios. Restablece el entorno de ejecución.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print('Revisión preparada:', commit)

## 3. Instalar y comprobar el código

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)
if subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout:
    raise RuntimeError('Las comprobaciones modificaron el repositorio temporal.')
print('Código y pruebas automáticas preparados.')

## 4. Validar formalmente el cierre rechazado de `v0.3`

In [ ]:
v03_summary = json.loads(v03_summary_path.read_text(encoding='utf-8'))
v03_record = json.loads(v03_record_path.read_text(encoding='utf-8'))
if v03_summary.get('status') != 'rejected_after_blinded_review':
    raise RuntimeError('v0.3 no quedó registrada como rechazada.')
if v03_summary.get('reviewed_knees') != 20 or v03_summary.get('acceptable_crops') != 0:
    raise RuntimeError('El cierre de v0.3 no contiene las veinte revisiones rechazadas.')
if v03_summary.get('rejected_crops') != 20 or v03_summary.get('technical_exclusions') != 0:
    raise RuntimeError('Los conteos del cierre de v0.3 no coinciden con la revisión registrada.')
if v03_summary.get('parameters_frozen') or not v03_summary.get('reviewed_without_outcome'):
    raise RuntimeError('El cierre de v0.3 no conserva el rechazo ciego y sin congelamiento.')
if v03_record.get('status') != v03_summary.get('status') or v03_record.get('reviewed_knees') != 20:
    raise RuntimeError('El registro y el resumen de cierre de v0.3 no concuerdan.')
for key in ('mass_processing_executed', 'partitions_created', 'training_executed', 'reserved_test_opened'):
    if v03_record.get(key) is not False:
        raise RuntimeError(f'El cierre de v0.3 no mantiene bloqueada la operación {key}.')
print('Cierre v0.3 validado: 0 aceptables, 20 rechazadas y parámetros no congelados.')

## 5. Ejecutar `tibiofemoral_crop_v0.4_pilot`

In [ ]:
config_path = REPO_DIR / 'configs' / 'joint_localization.v0.4.example.json'
subprocess.run([sys.executable, '-m', 'knee.joint_localization', '--config', str(config_path)], cwd=REPO_DIR, check=True)

## 6. Mostrar las veinte nuevas vistas ciegas

In [ ]:
from IPython.display import Image as IPImage, display

v04_dir = step4_dir / 'v0_4_piloto'
preview_dir = v04_dir / 'previews_ciegas'
previews = sorted(preview_dir.glob('case_*_localizacion.png'))
if len(previews) != 20:
    raise RuntimeError(f'Se esperaban 20 vistas y se encontraron {len(previews)}.')
for preview in previews:
    print(preview.stem)
    display(IPImage(filename=str(preview), width=800))
print('Nueva revisión pendiente en:', v04_dir / 'revision_visual_ciega.csv')

## 7. Registrar la repetición controlada

In [ ]:
from datetime import datetime, timezone

summary = json.loads((v04_dir / 'resumen_localizacion_publico.json').read_text(encoding='utf-8'))
if summary.get('status') != 'ready_for_blinded_review':
    raise RuntimeError('v0.4 no quedó lista para revisión.')
if summary.get('processed_unique_studies') != 10 or summary.get('processed_knees') != 20:
    raise RuntimeError('v0.4 no procesó exactamente las mismas diez adquisiciones y veinte rodillas.')
record = {
    'phase': 1,
    'step': 4,
    'operation': 'repeat_blinded_tibiofemoral_localization_v04_pilot',
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': commit,
    'status': summary['status'],
    'algorithm_version': summary['algorithm_version'],
    'localization_strategy': summary['localization_strategy'],
    'processed_unique_studies': summary['processed_unique_studies'],
    'processed_knees': summary['processed_knees'],
    'prior_version_status': v03_summary['status'],
    'visual_review_status': 'pending',
    'mass_processing_executed': False,
    'partitions_created': False,
    'training_executed': False,
    'reserved_test_opened': False,
}
(v04_dir / 'registro_preparacion.json').write_text(
    json.dumps(record, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
print('v0.4 preparada para revisión visual ciega. El paso 4 continúa abierto.', record)